<a href="https://colab.research.google.com/github/wswgadaniel-stack/proyecto_AD_c2_b-sico/blob/main/VentureOne.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
from collections import Counter
import csv
from datetime import date
import json
import math
import os
from pathlib import Path
import statistics
import subprocess
import sys
import unicodedata
import yaml

# --- 1. Ubicarse en la raíz del repositorio ---
REPO_URL = "https://github.com/solhecoo/ventureone-reto-ficha.git"
CARPETA = "ventureone-reto-ficha"

if "google.colab" in sys.modules and not os.path.exists(CARPETA):
  subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
if os.path.exists(CARPETA):
  os.chdir(CARPETA)
elif os.path.exists("../config/parametros.yaml"):
  os.chdir("..")

RAIZ = Path(".").resolve()
assert (RAIZ / "config" / "parametros.yaml").exists(), (
    "No se encontró config/parametros.yaml"
)

# --- 2. Cargar parámetros y capas ---
PARAMS = yaml.safe_load(
    (RAIZ / "config" / "parametros.yaml").read_text(encoding="utf-8")
)
with (RAIZ / "data" / "anuncios_muestra.csv").open(encoding="utf-8") as f:
  ANUNCIOS_RAW = list(csv.DictReader(f))
with (RAIZ / "data" / "inmuebles_prueba.csv").open(encoding="utf-8") as f:
  INMUEBLES_RAW = list(csv.DictReader(f))

TRANSPORTE = json.loads(
    (RAIZ / "data" / "geo" / "transporte.geojson").read_text(encoding="utf-8")
)
EQUIPAMIENTOS = json.loads(
    (RAIZ / "data" / "geo" / "equipamientos.geojson").read_text(encoding="utf-8")
)
RUTA_POT = RAIZ / PARAMS["norma"]["archivo_pot"]
POT = (
    json.loads(RUTA_POT.read_text(encoding="utf-8"))
    if RUTA_POT.exists()
    else None
)

RADIO_TIERRA_M = 6_371_008.8


# --- 3. Geometría ---
def haversine_m(lat1, lon1, lat2, lon2):
  p1, p2 = math.radians(lat1), math.radians(lat2)
  dp, dl = p2 - p1, math.radians(lon2 - lon1)
  a = (
      math.sin(dp / 2) ** 2
      + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
  )
  return 2 * RADIO_TIERRA_M * math.asin(math.sqrt(a))


def _en_anillo(lon, lat, anillo):
  dentro, j = False, len(anillo) - 1
  for i in range(len(anillo)):
    xi, yi = anillo[i][0], anillo[i][1]
    xj, yj = anillo[j][0], anillo[j][1]
    if (yi > lat) != (yj > lat) and lon < (xj - xi) * (lat - yi) / (
        yj - yi
    ) + xi:
      dentro = not dentro
    j = i
  return dentro


def punto_en_geometria(lat, lon, geometria):
  if geometria["type"] == "Polygon":
    poligonos = [geometria["coordinates"]]
  elif geometria["type"] == "MultiPolygon":
    poligonos = geometria["coordinates"]
  else:
    raise ValueError(f"Geometría no soportada: {geometria['type']}")
  for exterior, *huecos in poligonos:
    if _en_anillo(lon, lat, exterior) and not any(
        _en_anillo(lon, lat, h) for h in huecos
    ):
      return True
  return False


# --- 4. Limpieza y deduplicación ---
def _texto_normalizado(texto):
  sin_tildes = (
      unicodedata.normalize("NFKD", str(texto or ""))
      .encode("ascii", "ignore")
      .decode()
  )
  return " ".join(sin_tildes.replace("?", " ").lower().split())


def normalizar_estado(texto, estados):
  limpio = _texto_normalizado(texto)
  for canonico, variantes in estados.items():
    if limpio in (_texto_normalizado(v) for v in variantes):
      return canonico
  return "sin_clasificar"


def _numero(valor):
  try:
    n = float(valor)
  except (TypeError, ValueError):
    return None
  return n if n > 0 else None


def _preparar(fila, params):
  area, precio = _numero(fila.get("area_m2")), _numero(fila.get("precio"))
  return {
      **fila,
      "area_m2": area,
      "precio": precio,
      "precio_m2": precio / area if area and precio else None,
      "estado_canonico": normalizar_estado(
          fila.get("estado"), params["estados"]
      ),
  }


def _motivo_invalido(anuncio, params):
  if anuncio["area_m2"] is None:
    return "sin_area"
  if anuncio["precio"] is None:
    return "sin_precio"
  minimo, maximo = params["comparables"]["rango_precio_m2_valido"]
  if not minimo <= anuncio["precio_m2"] <= maximo:
    return "precio_m2_fuera_de_rango"
  if anuncio["estado_canonico"] == "sin_clasificar":
    return "estado_sin_clasificar"
  if anuncio.get("tipo_precio") not in params["comparables"]["tipos_precio"]:
    return "tipo_precio_no_aceptado"
  return None


def _clave_sin_id(anuncio, columnas):
  valores = tuple(anuncio.get(c) for c in columnas)
  return None if any(v in (None, "") for v in valores) else valores


def _deduplicar(anuncios, params):
  columnas = params["deduplicacion"]["claves_sin_id"]
  unicos, por_id, por_clave = [], {}, {}
  for a in anuncios:
    clave = _clave_sin_id(a, columnas)
    destino = por_id.get(a["id_anuncio"]) if a["id_anuncio"] else None
    if destino is None and clave is not None:
      destino = por_clave.get(clave)
    if destino is None:
      destino = {**a, "origenes_agrupados": [], "ids_agrupados": []}
      unicos.append(destino)
    destino["origenes_agrupados"].append(a["origen"])
    if a["id_anuncio"] and a["id_anuncio"] not in destino["ids_agrupados"]:
      destino["ids_agrupados"].append(a["id_anuncio"])
    if a["id_anuncio"]:
      por_id.setdefault(a["id_anuncio"], destino)
    if clave is not None:
      por_clave.setdefault(clave, destino)
  return unicos


def limpiar_anuncios(anuncios, params):
  preparados = [_preparar(fila, params) for fila in anuncios]
  unicos = _deduplicar(preparados, params)
  for a in unicos:
    a["motivo_invalido"] = _motivo_invalido(a, params)
  return unicos


# --- 5. Mercado ---
def _es_el_mismo_inmueble(anuncio, inmueble):
  return inmueble["origen"] in anuncio["origenes_agrupados"] or (
      inmueble.get("id_anuncio")
      and inmueble["id_anuncio"] in anuncio["ids_agrupados"]
  )


def _estadisticas(precios_m2):
  n = len(precios_m2)
  if n == 0:
    return {
        "n": 0,
        "mediana": None,
        "promedio": None,
        "minimo": None,
        "maximo": None,
        "desviacion": None,
        "coef_variacion": None,
    }
  promedio = statistics.mean(precios_m2)
  desviacion = statistics.stdev(precios_m2) if n >= 2 else None
  return {
      "n": n,
      "mediana": round(statistics.median(precios_m2)),
      "promedio": round(promedio),
      "minimo": round(min(precios_m2)),
      "maximo": round(max(precios_m2)),
      "desviacion": round(desviacion) if desviacion is not None else None,
      "coef_variacion": (
          round(desviacion / promedio, 4) if desviacion is not None else None
      ),
  }


def _nivel_confianza(n, coef_variacion, reglas):
  for nivel in ("alta", "media"):
    r = reglas[nivel]
    if (
        n >= r["min_comparables"]
        and coef_variacion is not None
        and coef_variacion <= r["max_coef_variacion"]
    ):
      return nivel, (
          f"{n} comparables (mínimo {r['min_comparables']}) y coeficiente de"
          f" variación {coef_variacion:.2%} (máximo"
          f" {r['max_coef_variacion']:.0%})"
      )
  cv = "sin dato" if coef_variacion is None else f"{coef_variacion:.2%}"
  m = reglas["media"]
  return "baja", (
      f"{n} comparables y coeficiente de variación {cv}: no cumple el nivel"
      f" media (mínimo {m['min_comparables']} comparables, máximo"
      f" {m['max_coef_variacion']:.0%})"
  )


def _resumen_por_estado(inmueble, anuncios_limpios, params):
  resumen = {}
  for estado in params["estados"]:
    valores = [
        a["precio_m2"]
        for a in anuncios_limpios
        if a["motivo_invalido"] is None
        and a["zona"] == inmueble["zona"]
        and a["estado_canonico"] == estado
        and not _es_el_mismo_inmueble(a, inmueble)
    ]
    e = _estadisticas(valores)
    resumen[estado] = {"n": e["n"], "mediana": e["mediana"]}
  return resumen


def bloque_mercado(inmueble, anuncios_limpios, params):
  tol = params["comparables"]["tolerancia_area"]
  area_inm = float(inmueble["area_m2"])
  estado_inm = inmueble.get("estado_canonico") or normalizar_estado(
      inmueble["estado"], params["estados"]
  )
  comparables = [
      a
      for a in anuncios_limpios
      if a["motivo_invalido"] is None
      and a["zona"] == inmueble["zona"]
      and a["estado_canonico"] == estado_inm
      and abs(a["area_m2"] - area_inm) <= tol * area_inm
      and not _es_el_mismo_inmueble(a, inmueble)
  ]
  est = _estadisticas([a["precio_m2"] for a in comparables])
  nivel, motivo = _nivel_confianza(
      est["n"], est["coef_variacion"], params["confianza"]
  )
  return {
      "estado_referencia": estado_inm,
      "precio_m2": est,
      "confianza": {"nivel": nivel, "motivo": motivo},
      "comparables": [
          {
              "origen": a["origen"],
              "id_anuncio": a["id_anuncio"],
              "fuente": a["fuente"],
              "area_m2": a["area_m2"],
              "precio": round(a["precio"]),
              "precio_m2": round(a["precio_m2"]),
          }
          for a in sorted(comparables, key=lambda x: x["precio_m2"])
      ],
      "precio_m2_por_estado_en_zona": _resumen_por_estado(
          inmueble, anuncios_limpios, params
      ),
  }


# --- 6. Entorno y Norma ---
def bloque_entorno(inmueble, transporte, equipamientos, params):
  p_ent = params["entorno"]
  lat, lon = float(inmueble["lat"]), float(inmueble["lon"])
  precision = inmueble["precision_coordenadas"]
  radio = p_ent["radio_busqueda_m"]
  features_transporte = (
      transporte["features"] if isinstance(transporte, dict) else transporte
  )
  features_equip = (
      equipamientos["features"]
      if isinstance(equipamientos, dict)
      else equipamientos
  )

  estaciones = sorted(
      (
          (
              haversine_m(
                  lat,
                  lon,
                  f["geometry"]["coordinates"][1],
                  f["geometry"]["coordinates"][0],
              ),
              f,
          )
          for f in features_transporte
          if f["properties"]["sistema"] in p_ent["sistemas_transporte"]
      ),
      key=lambda par: par[0],
  )
  cercana = None
  if estaciones:
    d, f = estaciones[0]
    cercana = {
        "nombre": f["properties"]["nombre"],
        "sistema": f["properties"]["sistema"],
        "distancia_m": round(d),
    }

  conteo = {cat: 0 for cat in p_ent["categorias_equipamientos"]}
  for f in features_equip:
    cat = f["properties"]["categoria"]
    if cat in conteo:
      d = haversine_m(
          lat,
          lon,
          f["geometry"]["coordinates"][1],
          f["geometry"]["coordinates"][0],
      )
      if d <= radio:
        conteo[cat] += 1

  return {
      "radio_busqueda_m": radio,
      "estacion_mas_cercana": cercana,
      "estaciones_en_radio": sum(1 for d, _ in estaciones if d <= radio),
      "equipamientos_en_radio": conteo,
      "aproximado": precision != "cruce",
  }


def bloque_norma(lat, lon, municipio, capa_pot, params_norma):
  if municipio not in params_norma["municipios_cubiertos"]:
    return {
        "tratamiento_pot": None,
        "estado": "no_disponible",
        "motivo": (
            "La capa POT cubre solo:"
            f" {', '.join(params_norma['municipios_cubiertos'])}"
        ),
    }
  if capa_pot is None:
    return {
        "tratamiento_pot": None,
        "estado": "pendiente_capa",
        "motivo": f"Falta el archivo {params_norma['archivo_pot']}",
    }
  campo = params_norma["campo_tratamiento"]
  for f in capa_pot["features"]:
    if f.get("geometry") and punto_en_geometria(lat, lon, f["geometry"]):
      return {
          "tratamiento_pot": f["properties"].get(campo),
          "estado": "encontrado",
          "motivo": "",
      }
  return {
      "tratamiento_pot": None,
      "estado": "fuera_de_capa",
      "motivo": "El punto no cae en ningún polígono",
  }


# --- 7. Construcción de ficha y ejecución ---
CRUDOS_PREPARADOS = [_preparar(r, PARAMS) for r in ANUNCIOS_RAW]
ANUNCIOS_LIMPIOS = limpiar_anuncios(ANUNCIOS_RAW, PARAMS)


def _calidad_zona(zona):
  crudos_z = [a for a in CRUDOS_PREPARADOS if a["zona"] == zona]
  unicos_z = [a for a in ANUNCIOS_LIMPIOS if a["zona"] == zona]
  invalidos = Counter(
      a["motivo_invalido"] for a in unicos_z if a["motivo_invalido"]
  )
  return {
      "anuncios_en_zona": len(crudos_z),
      "duplicados_eliminados": len(crudos_z) - len(unicos_z),
      "anuncios_unicos": len(unicos_z),
      "anuncios_invalidos": dict(sorted(invalidos.items())),
      "anuncios_validos": len(unicos_z) - sum(invalidos.values()),
  }


def _datos_faltantes(inmueble, mercado):
  faltantes = []
  if not inmueble.get("direccion"):
    faltantes.append("direccion")
  if inmueble.get("precision_coordenadas") != "cruce":
    faltantes.append("coordenadas_exactas")
  origenes_comp = {c["origen"] for c in mercado["comparables"]}
  comps_anuncios = [
      a for a in ANUNCIOS_LIMPIOS if a["origen"] in origenes_comp
  ]
  if not any(a.get("fecha") for a in comps_anuncios):
    faltantes.append("fecha_publicacion_comparables")
  if not any(a.get("tipo_precio") == "cierre" for a in comps_anuncios):
    faltantes.append("precios_de_cierre")
  return faltantes


def generar_ficha(inmueble):
  estado_canonico = normalizar_estado(inmueble["estado"], PARAMS["estados"])
  inm = {
      **inmueble,
      "area_m2": float(inmueble["area_m2"]),
      "lat": float(inmueble["lat"]),
      "lon": float(inmueble["lon"]),
      "estado_canonico": estado_canonico,
  }
  mercado = bloque_mercado(inm, ANUNCIOS_LIMPIOS, PARAMS)
  return {
      "id_inmueble": inm["id_inmueble"],
      "inmueble": {
          "direccion": inm.get("direccion") or None,
          "barrio": inm["barrio"],
          "municipio": inm["municipio"],
          "zona": inm["zona"],
          "area_m2": inm["area_m2"],
          "estado_texto": inm["estado"],
          "estado": estado_canonico,
          "coordenadas": {
              "lat": inm["lat"],
              "lon": inm["lon"],
              "precision": inm["precision_coordenadas"],
              "fuente": inm["fuente_coordenadas"],
          },
      },
      "mercado": mercado,
      "calidad_datos_zona": _calidad_zona(inm["zona"]),
      "entorno": bloque_entorno(inm, TRANSPORTE, EQUIPAMIENTOS, PARAMS),
      "norma": bloque_norma(
          inm["lat"], inm["lon"], inm["municipio"], POT, PARAMS["norma"]
      ),
      "datos_faltantes": _datos_faltantes(inm, mercado),
      "metadatos": {
          "version_parametros": PARAMS["version"],
          "fecha_generacion": date.today().isoformat(),
          "fecha_capas_osm": TRANSPORTE["metadata"]["fecha_osm"],
          "fuentes": [
              "data/anuncios_muestra.csv",
              "OpenStreetMap (ODbL)",
          ] + ([PARAMS["norma"]["archivo_pot"]] if POT else []),
      },
  }


carpeta_salida = RAIZ / "salidas" / "participante"
carpeta_salida.mkdir(parents=True, exist_ok=True)
for fila_inm in INMUEBLES_RAW:
  ficha = generar_ficha(fila_inm)
  (carpeta_salida / f"{fila_inm['id_inmueble']}.json").write_text(
      json.dumps(ficha, ensure_ascii=False, indent=2), encoding="utf-8"
  )

print("Fichas generadas:", sorted(os.listdir(carpeta_salida)))
if (RAIZ / "validacion" / "validar.py").exists():
  val = subprocess.run(
      [sys.executable, "validacion/validar.py", str(carpeta_salida)],
      capture_output=True,
      text=True,
  )
  print(val.stdout or val.stderr)

Fichas generadas: ['INM-01.json', 'INM-02.json', 'INM-03.json', 'INM-04.json', 'INM-05.json', 'INM-06.json', 'INM-07.json', 'INM-08.json', 'INM-09.json', 'INM-10.json', 'INM-11.json']
✓ INM-04
✓ INM-06
✓ INM-09



In [11]:
# @title Ver ficha del inmueble dinámicamente
import json
import os
from pathlib import Path
from IPython.display import HTML, display

id_buscar = "INM-01"

# Buscamos la ruta de manera absoluta garantizando que apunte dentro de la carpeta actual del proceso (el repositorio)
ruta_actual = Path(os.getcwd()).resolve()
if ruta_actual.name == "ventureone-reto-ficha":
    ruta_json = ruta_actual / "salidas" / "participante" / f"{id_buscar}.json"
else:
    ruta_json = ruta_actual / "ventureone-reto-ficha" / "salidas" / "participante" / f"{id_buscar}.json"

ver_json_completo = True

posibles_campos_id = ["id_inmueble", "id", "codigo", "id_propiedad", "inmueble_id"]

try:
    with open(ruta_json, "r", encoding="utf-8") as f:
        datos = json.load(f)
except FileNotFoundError:
    print(f"❌ No se encontró el archivo en la ruta generada: {ruta_json}")
    print("Verifica que el identificador ingresado en 'id_buscar' sea el correcto y corresponda a una de tus fichas generadas.")
    datos = None

def extraer_id(item):
    if not isinstance(item, dict):
        return None
    for campo in posibles_campos_id:
        if campo in item and item[campo] is not None:
            return str(item[campo])
    return None

ficha = None
if datos is not None:
    if isinstance(datos, dict):
        if extraer_id(datos) == str(id_buscar):
            ficha = datos
        elif id_buscar in datos and isinstance(datos[id_buscar], dict):
            ficha = datos[id_buscar]
        else:
            for v in datos.values():
                if isinstance(v, list):
                    ficha = next((x for x in v if extraer_id(x) == str(id_buscar)), None)
                    if ficha:
                        break
    elif isinstance(datos, list):
        ficha = next((x for x in datos if extraer_id(x) == str(id_buscar)), None)

if datos is not None and not ficha:
    print(f"⚠️ No se encontró el ID '{id_buscar}' en '{ruta_json}'.")
elif ficha:
    id_mostrado = extraer_id(ficha) or id_buscar
    m = ficha.get("mercado", {})
    e = ficha.get("entorno", {})

    html = f"""
    <div style='border:1px solid #cbd5e1; border-radius:10px; padding:16px; max-width:540px; font-family:Arial, sans-serif; background:#f8fafc;'>
        <h3 style='margin:0 0 6px 0; color:#0f172a;'>{ficha.get('inmueble', {}).get('barrio') or f'Inmueble {id_mostrado}'}</h3>
        <div style='color:#475569; font-size:13px; margin-bottom:12px;'>
            ID: <b>{id_mostrado}</b> · {ficha.get('inmueble', {}).get('municipio', 'N/A')}
        </div>
        <p style='margin:4px 0;'><b>Estado:</b> {ficha.get('inmueble', {}).get('estado_texto', 'N/A')} | <b>Área:</b> {ficha.get('inmueble', {}).get('area_m2', 'N/A')} m²</p>
        <p style='margin:4px 0;'><b>Confianza de Mercado:</b> {m.get('confianza', {}).get('nivel', 'N/A')}</p>
        <p style='margin:4px 0;'><b>Estación cercana:</b> {e.get('estacion_mas_cercana', {}).get('nombre', 'Ninguna')} ({e.get('estacion_mas_cercana', {}).get('distancia_m', 'N/A')} m)</p>
    </div>
    """
    display(HTML(html))

    if ver_json_completo:
        print("\n--- Datos completos de la ficha (JSON) ---")
        print(json.dumps(ficha, indent=2, ensure_ascii=False))


--- Datos completos de la ficha (JSON) ---
{
  "id_inmueble": "INM-01",
  "inmueble": {
    "direccion": "Cra 36A con Calle 41",
    "barrio": "El Salvador",
    "municipio": "medellin",
    "zona": "buenos_aires_el_salvador",
    "area_m2": 73.0,
    "estado_texto": "Remodelar",
    "estado": "por_remodelar",
    "coordenadas": {
      "lat": 6.238961,
      "lon": -75.560963,
      "precision": "cruce",
      "fuente": "OSM cruce Carrera 36A x Calle 41 (separación 0 m)"
    }
  },
  "mercado": {
    "estado_referencia": "por_remodelar",
    "precio_m2": {
      "n": 10,
      "mediana": 4555556,
      "promedio": 4250665,
      "minimo": 2247191,
      "maximo": 5214286,
      "desviacion": 943969,
      "coef_variacion": 0.2221
    },
    "confianza": {
      "nivel": "media",
      "motivo": "10 comparables (mínimo 3) y coeficiente de variación 22.21% (máximo 25%)"
    },
    "comparables": [
      {
        "origen": "extension_poligono.xlsx:REMODELAR:73",
        "id_anuncio": "